In [28]:
from datetime import datetime
import unicodedata
import numpy as np
import random
import os
import pandas as pd
import string

from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output, State
from dash import callback_context
import plotly.graph_objs as go
import plotly.graph_objects as go2

from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from fpdf import FPDF

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

import tempfile
import shutil

# -------------------------------------------------
# Константы / настройки
# -------------------------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

R_RATIO = 2
T_MAX = 260
DT = 0.01
STEP = 50
WINDOW = int(2 / DT)
DELAY_STEPS = int(0.5 / DT)
PHASE_THRESH = 0.15
SYNC_DETECT_STEP = int(35 / DT)

SIGMA = 10.0
RHO = 28.0
BETA = 8.0 / 3.0

NOISE_TMAX = 200
NOISE_DT = 0.01

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------------------------------------
# Утилиты: unicode, алфавиты
# -------------------------------------------------
def valid_unicode_char(cp):
    try:
        ch = chr(cp)
        if ch.isprintable() and not ch.isspace():
            if 'UNASSIGNED' not in unicodedata.name(ch, ''):
                return True
    except Exception:
        return False
    return False


def get_digits():
    return list(string.digits)


def get_ascii_letters():
    return list(string.ascii_letters)


def get_cyrillic():
    return [chr(cp) for cp in range(0x0400, 0x0500) if valid_unicode_char(cp)]


def get_ascii_symbols():
    return [ch for ch in (chr(i) for i in range(32, 127))
            if not ch.isalnum() and valid_unicode_char(ord(ch))]


def get_unicode_symbols():
    return [chr(cp) for cp in range(32, 0x110000) if valid_unicode_char(cp)]


def get_ebcdic_symbols():
    ebcdic_codes = list(range(64, 91)) + list(range(97, 123))
    return [chr(i) for i in ebcdic_codes if valid_unicode_char(i)]


def get_koi8_symbols():
    return [chr(cp) for cp in range(0xC0, 0x100) if valid_unicode_char(cp)]


def select_alphabet(choice):
    if choice == "digits":
        return get_digits()
    elif choice == "latin":
        return get_ascii_letters()
    elif choice == "cyrillic":
        return get_cyrillic()
    elif choice == "ascii":
        return [chr(i) for i in range(32, 127) if valid_unicode_char(i)]
    elif choice == "unicode":
        return get_unicode_symbols()
    elif choice == "ebcdic":
        return get_ebcdic_symbols()
    elif choice == "koi8":
        return get_koi8_symbols()
    elif choice == "special":
        return get_ascii_symbols()
    else:
        return [chr(i) for i in range(32, 127)]


# -------------------------------------------------
# Лоренц: генераторы и численные методы
# -------------------------------------------------
def lorenz_noise(t_max, dt, sigma=10., rho=28., beta=8 / 3, x0=1., y0=1., z0=1.):
    n_steps = int(t_max / dt)
    x = np.zeros(n_steps)
    y = np.zeros(n_steps)
    z = np.zeros(n_steps)
    x[0], y[0], z[0] = x0, y0, z0
    for i in range(1, n_steps):
        dx = sigma * (y[i - 1] - x[i - 1])
        dy = x[i - 1] * (rho - z[i - 1]) - y[i - 1]
        dz = x[i - 1] * y[i - 1] - beta * z[i - 1]
        x[i] = x[i - 1] + dx * dt
        y[i] = y[i - 1] + dy * dt
        z[i] = z[i - 1] + dz * dt
    return x, y, z


class LorenzGenerator:
    def __init__(self, t_max, dt, x0, y0, z0, r=None, phase_shift=0.0):
        self.t_max = t_max
        self.dt = dt
        self.r = r
        self.x0 = x0 + phase_shift
        self.y0 = y0 + phase_shift
        self.z0 = z0 + phase_shift

    def run(self):
        x, y, z = lorenz_noise(self.t_max, self.dt, x0=self.x0, y0=self.y0, z0=self.z0)
        if self.r is not None:
            cutoff = int(0.5 * len(x))
            x[cutoff:] = x[cutoff - 1]
            y[cutoff:] = y[cutoff - 1]
            z[cutoff:] = z[cutoff - 1]
        return x, y, z


def dual_run(t_max, dt, r=None, phase_offset=0.0):
    g1 = LorenzGenerator(t_max, dt, 0.1, 0.0, 0.0, r=r, phase_shift=0.0)
    g2 = LorenzGenerator(t_max, dt, 0.1, 0.0, 0.0, r=r, phase_shift=phase_offset)
    x1, y1, z1 = g1.run()
    x2, y2, z2 = g2.run()
    return (x1, y1, z1), (x2, y2, z2)


def lorenz_deriv(x, y, z, sigma, rho, beta, r, x_other):
    dx = sigma * (y - x) - r * (x - x_other)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return dx, dy, dz


def runge_kutta_step(x, y, z, dt, sigma, rho, beta, r, x_other):
    dx1, dy1, dz1 = lorenz_deriv(x, y, z, sigma, rho, beta, r, x_other)
    dx2, dy2, dz2 = lorenz_deriv(
        x + dx1 * dt / 2,
        y + dy1 * dt / 2,
        z + dz1 * dt / 2,
        sigma, rho, beta, r, x_other
    )
    dx3, dy3, dz3 = lorenz_deriv(
        x + dx2 * dt / 2,
        y + dy2 * dt / 2,
        z + dz2 * dt / 2,
        sigma, rho, beta, r, x_other
    )
    dx4, dy4, dz4 = lorenz_deriv(
        x + dx3 * dt,
        y + dy3 * dt,
        z + dz3 * dt,
        sigma, rho, beta, r, x_other
    )

    x_new = x + (dx1 + 2 * dx2 + 2 * dx3 + dx4) * dt / 6
    y_new = y + (dy1 + 2 * dy2 + 2 * dy3 + dy4) * dt / 6
    z_new = z + (dz1 + 2 * dz2 + 2 * dz3 + dz4) * dt / 6
    return x_new, y_new, z_new


def lorenz_coupled(t_max, dt, x01, y01, z01, x02, y02, z02, r):
    n_steps = int(t_max / dt)
    x1 = np.zeros(n_steps)
    y1 = np.zeros(n_steps)
    z1 = np.zeros(n_steps)
    x2 = np.zeros(n_steps)
    y2 = np.zeros(n_steps)
    z2 = np.zeros(n_steps)

    x1[0], y1[0], z1[0] = x01, y01, z01
    x2[0], y2[0], z2[0] = x02, y02, z02

    for i in range(1, n_steps):
        x2_prev = x2[i - 1] if i - 1 >= DELAY_STEPS else x02
        x1[i], y1[i], z1[i] = runge_kutta_step(
            x1[i - 1], y1[i - 1], z1[i - 1],
            dt, SIGMA, RHO, BETA, r, x2_prev
        )
        if i < DELAY_STEPS:
            x2[i], y2[i], z2[i] = x02, y02, z02
        else:
            x2[i], y2[i], z2[i] = runge_kutta_step(
                x2[i - 1], y2[i - 1], z2[i - 1],
                dt, SIGMA, RHO, BETA, r, x1[i - 1]
            )

    return (x1, y1, z1), (x2, y2, z2)


# -------------------------------------------------
# Аналитика по символам и спектру
# -------------------------------------------------
def chaos_signal_to_sequence(signal, alphabet):
    signal = np.asarray(signal)
    minv, maxv = np.min(signal), np.max(signal)
    if maxv - minv == 0:
        idx = np.zeros_like(signal, dtype=int)
    else:
        n_chars = len(alphabet)
        idx = ((signal - minv) / (maxv - minv) * (n_chars - 1)).astype(int)
    return [alphabet[i] for i in idx]


def save_sequence_csv(seq, prefix="sequence"):
    filename = os.path.join(
        OUTPUT_DIR,
        f"{prefix}_{datetime.now().strftime('%d.%m.%Y_%H_%M_%S')}.csv"
    )
    df = pd.DataFrame({'Sequence': seq})
    df.to_csv(filename, index=False, encoding='utf-8')
    return filename, len(seq)


def export_to_excel(seq, t_max, dt, r_ratio, H1, H2, mean_match, sync_time, prefix="sequence"):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Data"

    ws['A1'] = 'Sequence'
    ws['B1'] = 'Parameters'
    ws['A1'].font = Font(bold=True)
    ws['B1'].font = Font(bold=True)
    gray = PatternFill(start_color="CCCCCC", end_color="CCCCCC", fill_type="solid")
    ws['A1'].fill = gray
    ws['B1'].fill = gray

    for idx, symbol in enumerate(seq, start=2):
        ws[f'A{idx}'] = symbol

    params_list = [
        f"T_MAX: {t_max}",
        f"DT: {dt}",
        f"R_RATIO: {r_ratio}",
        f"Entropy G1: {H1:.3f}",
        f"Entropy G2: {H2:.3f}",
        f"Mean Match: {mean_match:.2f}%",
        f"Sync Time: {sync_time:.2f}s" if sync_time is not None else "Sync Time: Not detected"
    ]
    for idx, param in enumerate(params_list, start=2):
        ws[f'B{idx}'] = param
        ws[f'B{idx}'].alignment = Alignment(wrap_text=True)

    ws.column_dimensions['A'].width = 15
    ws.column_dimensions['B'].width = 40

    filename = os.path.join(
        OUTPUT_DIR,
        f"{prefix}_{datetime.now().strftime('%d.%m.%Y_%H_%M_%S')}.xlsx"
    )
    wb.save(filename)
    return filename


def export_to_word_detailed(seq1, seq2, stat1, stat2, H1, H2,
                            t_max, dt, r_ratio, mean_match, sync_time_s,
                            mode_desc, symbol_of_interest, alphabet_size,
                            prefix="sequence"):
    doc = Document()

    title = doc.add_heading('Отчёт анализа хаотических систем Лоренца', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    subtitle = doc.add_paragraph(datetime.now().strftime('Дата отчёта: %d.%m.%Y %H:%M:%S'))
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    subtitle_mode = doc.add_paragraph(f'Режим: {mode_desc}')
    subtitle_mode.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_heading('1. Параметры моделирования', level=1)
    doc.add_paragraph(f"Режим анализа: {mode_desc}")
    doc.add_paragraph(f"Время интегрирования (T_MAX): {t_max} сек")
    doc.add_paragraph(f"Шаг интегрирования (DT): {dt}")
    doc.add_paragraph(f"Коэффициент связи (R_RATIO): {r_ratio}")
    doc.add_paragraph(
        f"Константы системы Лоренца: σ={SIGMA}, ρ={RHO}, β={BETA:.4f}"
    )
    total_steps = int(t_max / dt)
    doc.add_paragraph(f"Общее количество шагов интегрирования: {total_steps:,}")
    doc.add_paragraph(
        f"Точность численного интегрирования: метод Рунге-Кутты 4-го порядка с шагом {dt}"
    )

    doc.add_heading('2. Энтропийный анализ', level=1)
    doc.add_paragraph(
        f"Энтропия последовательности G1 (Генератор 1): {H1:.4f} бит/символ"
    )
    doc.add_paragraph(
        f"Энтропия последовательности G2 (Генератор 2): {H2:.4f} бит/символ"
    )
    doc.add_paragraph(f"Размер используемого алфавита: {alphabet_size} уникальных символов")

    max_entropy = np.log2(alphabet_size)
    doc.add_paragraph(
        f"Теоретическая максимальная энтропия для этого алфавита: {max_entropy:.4f} бит/символ "
        f"(при полностью равномерном распределении вероятностей)"
    )

    entropy_usage_g1 = (H1 / max_entropy) * 100
    entropy_usage_g2 = (H2 / max_entropy) * 100
    doc.add_paragraph(
        f"Коэффициент использования энтропии (G1): {entropy_usage_g1:.1f}% от максимума"
    )
    doc.add_paragraph(
        f"Коэффициент использования энтропии (G2): {entropy_usage_g2:.1f}% от максимума"
    )

    doc.add_paragraph()

    if H1 > 0.9 * max_entropy and H2 > 0.9 * max_entropy:
        p = doc.add_paragraph(
            "✓ ВЫВОД: Обе последовательности имеют ВЫСОКУЮ энтропию, близко к случайному "
            "распределению (>90% от максимума). Это указывает на высокую степень хаотичности "
            "и непредсказуемости. Такие последовательности подходят для криптографических применений."
        )
        p.runs[0].font.size = Pt(11)
        p.runs[0].font.bold = True
    elif H1 > 0.7 * max_entropy and H2 > 0.7 * max_entropy:
        p = doc.add_paragraph(
            "ℹ ВЫВОД: Энтропия УМЕРЕННАЯ (70-90% от максимума) — последовательности имеют "
            "структуру и не случайны, что типично для детерминированных хаотических систем. "
            "Несмотря на детерминированность уравнений, результирующие символьные последовательности "
            "имеют хаотический характер."
        )
        p.runs[0].font.size = Pt(11)
    else:
        p = doc.add_paragraph(
            "⚠ ВЫВОД: Энтропия НИЗКАЯ (<70% от максимума) — последовательности содержат "
            "значительную структурность и неслучайность. Некоторые символы или пары символов "
            "встречаются чаще других."
        )
        p.runs[0].font.size = Pt(11)

    doc.add_heading('3. Анализ синхронизации и совпадений символов', level=1)
    doc.add_paragraph(
        f"Метод оценки: скользящее окно размером {WINDOW} символов с шагом {STEP} позиций"
    )
    doc.add_paragraph(
        f"Средняя доля совпадающих символов в окне: {mean_match:.2f}%"
    )

    if sync_time_s is not None:
        sync_index = int(sync_time_s / dt)
        percent_reached = (sync_index / total_steps) * 100
        doc.add_paragraph(
            f"Время обнаружения полной синхронизации: {sync_time_s:.4f} сек "
            f"({sync_index:,} шагов, примерно {percent_reached:.1f}% от общего времени)"
        )
        p = doc.add_paragraph(
            "✓ СИНХРОНИЗАЦИЯ ДОСТИГНУТА: две системы генерируют идентичные символьные последовательности "
            "с момента синхронизации. Это демонстрирует, что несмотря на детерминированный хаос в каждой "
            "системе отдельно, при достаточной связи между ними их состояния совпадают."
        )
        p.runs[0].font.bold = True
        p.runs[0].font.color.rgb = RGBColor(0, 128, 0)
    else:
        p = doc.add_paragraph(
            "✗ СИНХРОНИЗАЦИЯ НЕ ОБНАРУЖЕНА в течение всего периода интегрирования. "
            "Системы не сгенерировали окно идентичных символов требуемого размера. "
            "Возможные причины: коэффициент связи R слишком мал, время моделирования недостаточно, "
            "или начальные условия системы не позволяют синхронизации."
        )
        p.runs[0].font.bold = True
        p.runs[0].font.color.rgb = RGBColor(255, 0, 0)

    doc.add_paragraph()

    if mean_match > 80:
        doc.add_paragraph(
            f"Интерпретация средней доли ({mean_match:.1f}%): ВЫСОКАЯ корреляция между двумя системами. "
            f"Системы находятся в состоянии близком к синхронизации и демонстрируют согласованное поведение."
        )
    elif mean_match > 50:
        doc.add_paragraph(
            f"Интерпретация средней доли ({mean_match:.1f}%): УМЕРЕННАЯ корреляция между двумя системами. "
            f"Системы находятся в процессе приближения к синхронизации, но ещё не полностью синхронизированы."
        )
    elif mean_match > 30:
        doc.add_paragraph(
            f"Интерпретация средней доли ({mean_match:.1f}%): СЛАБАЯ корреляция между двумя системами. "
            f"Системы генерируют в основном независимые последовательности с редкими совпадениями."
        )
    else:
        doc.add_paragraph(
            f"Интерпретация средней доли ({mean_match:.1f}%): ОЧЕНЬ СЛАБАЯ или ОТСУТСТВУЮЩАЯ корреляция. "
            f"Системы функционируют практически независимо друг от друга."
        )

    doc.add_heading('4. Статистика повтора конкретного символа', level=1)
    doc.add_paragraph(f"Анализируемый символ: '{symbol_of_interest}'")
    doc.add_paragraph(
        f"Эта статистика показывает, как часто выбранный символ появляется в последовательности "
        f"и с какой регулярностью."
    )

    if stat1:
        p1 = doc.add_paragraph()
        p1.add_run(f"Генератор 1 (G1):").bold = True
        doc.add_paragraph(f"Количество появлений символа '{symbol_of_interest}': {stat1['count']}")
        doc.add_paragraph(
            f"Среднее время между повторами: {stat1['mean_interval_s']:.4f} сек "
            f"(в среднем каждые {stat1['mean_interval_s']:.2f} секунды)"
        )
        doc.add_paragraph(
            f"Стандартное отклонение интервалов: {stat1['std_interval_s']:.4f} сек "
            f"(показывает разброс времён между повторениями)"
        )
        cv = (stat1['std_interval_s'] / stat1['mean_interval_s']) if stat1['mean_interval_s'] > 0 else 0
        doc.add_paragraph(
            f"Коэффициент вариации: {cv:.3f} "
            f"({'регулярные' if cv < 0.5 else 'умеренно нерегулярные' if cv < 1.0 else 'сильно нерегулярные'} интервалы)"
        )
    else:
        doc.add_paragraph(
            f"Генератор 1 (G1): символ '{symbol_of_interest}' появляется менее чем 2 раза. "
            f"Недостаточно данных для вычисления статистики повторений."
        )

    if stat2:
        p2 = doc.add_paragraph()
        p2.add_run(f"Генератор 2 (G2):").bold = True
        doc.add_paragraph(f"Количество появлений символа '{symbol_of_interest}': {stat2['count']}")
        doc.add_paragraph(
            f"Среднее время между повторами: {stat2['mean_interval_s']:.4f} сек "
            f"(в среднем каждые {stat2['mean_interval_s']:.2f} секунды)"
        )
        doc.add_paragraph(
            f"Стандартное отклонение интервалов: {stat2['std_interval_s']:.4f} сек"
        )
        cv = (stat2['std_interval_s'] / stat2['mean_interval_s']) if stat2['mean_interval_s'] > 0 else 0
        doc.add_paragraph(
            f"Коэффициент вариации: {cv:.3f} "
            f"({'регулярные' if cv < 0.5 else 'умеренно нерегулярные' if cv < 1.0 else 'сильно нерегулярные'} интервалы)"
        )
    else:
        doc.add_paragraph(
            f"Генератор 2 (G2): символ '{symbol_of_interest}' появляется менее чем 2 раза. "
            f"Недостаточно данных для вычисления статистики повторений."
        )

    doc.add_heading('5. Физическая интерпретация динамики систем', level=1)
    doc.add_paragraph(
        f"Системы Лоренца описываются тремя нелинейными дифференциальными уравнениями первого порядка. "
        f"При определённых параметрах (в данном случае σ={SIGMA}, ρ={RHO}, β={BETA:.3f}) системы "
        f"демонстрируют детерминированный хаос — непредсказуемое поведение, несмотря на полную "
        f"детерминированность (отсутствие случайности в уравнениях)."
    )
    doc.add_paragraph(
        f"В трёхмерном фазовом пространстве (x, y, z) траектория системы образует странный аттрактор — "
        f"сложную фрактальную структуру, которую система «посещает» бесконечно много раз, но никогда "
        f"не повторяет точно (в силу хаотичности)."
    )
    doc.add_paragraph(
        f"Режим работы: {mode_desc}"
    )

    if "Coupled" in mode_desc or "coupled" in mode_desc:
        doc.add_paragraph(
            f"В режиме связанных систем две копии системы Лоренца взаимодействуют с коэффициентом связи R={r_ratio}. "
            f"Это взаимодействие может привести к явлению ХАОТИЧЕСКОЙ СИНХРОНИЗАЦИИ, когда две изначально "
            f"независимые хаотические системы, несмотря на детерминированный хаос в каждой из них, начинают "
            f"генерировать идентичные траектории. Пороговая величина связи R, требуемая для синхронизации, "
            f"зависит от структуры системы и параметров."
        )
    else:
        doc.add_paragraph(
            f"В режиме двойного запуска две системы Лоренца работают независимо с разными начальными условиями. "
            f"Несмотря на начальную близость начальных условий и идентичность параметров, траектории "
            f"экспоненциально расходятся из-за чувствительности к начальным условиям — фундаментального "
            f"свойства хаотических систем. Это демонстрирует, почему долгосрочное предсказание хаотических "
            f"систем практически невозможно."
        )

    doc.add_heading('6. Выводы и интерпретация результатов', level=1)

    findings = []

    if H1 > 0.9 * max_entropy and H2 > 0.9 * max_entropy:
        findings.append(
            f"• Обе системы генерируют последовательности с ВЫСОКОЙ энтропией ({H1:.3f} и {H2:.3f} бит/символ), "
            f"близко к теоретическому максимуму {max_entropy:.3f}. Это свидетельствует о хаотичности систем."
        )
    else:
        findings.append(
            f"• Энтропия последовательностей УМЕРЕННАЯ ({H1:.3f} и {H2:.3f} бит/символ), "
            f"что характерно для детерминированных хаотических систем."
        )

    if mean_match > 80:
        findings.append(
            f"• Высокая доля совпадений ({mean_match:.1f}%) указывает на СИЛЬНУЮ синхронизацию систем. "
            f"Они работают согласованно и предсказуемо относительно друг друга."
        )
    elif mean_match > 50:
        findings.append(
            f"• Умеренная доля совпадений ({mean_match:.1f}%) указывает на ЧАСТИЧНУЮ синхронизацию. "
            f"Системы находятся в промежуточном состоянии."
        )
    else:
        findings.append(
            f"• Низкая доля совпадений ({mean_match:.1f}%) указывает на то, что системы работают "
            f"в основном НЕЗАВИСИМО друг от друга."
        )

    if sync_time_s is not None:
        findings.append(
            f"• Полная синхронизация достигнута за {sync_time_s:.3f} сек, что составляет "
            f"примерно {(sync_time_s / t_max) * 100:.1f}% от общего времени моделирования."
        )
    else:
        findings.append(
            f"• Полная синхронизация НЕ произошла. Может потребоваться увеличение времени моделирования "
            f"или силы взаимодействия."
        )

    for finding in findings:
        doc.add_paragraph(finding, style='List Bullet')

    doc.add_heading('7. Рекомендации для дальнейшего анализа', level=1)
    doc.add_paragraph(
        "• Увеличить время моделирования для более полного наблюдения асимптотической динамики синхронизации",
        style='List Bullet'
    )
    doc.add_paragraph(
        "• Систематически варьировать коэффициент связи R для исследования переходов синхронизации",
        style='List Bullet'
    )
    doc.add_paragraph(
        "• Вычислить ляпуновские показатели для количественной оценки хаотичности систем",
        style='List Bullet'
    )
    doc.add_paragraph(
        "• Применить вейвлет-анализ для локального временно-частотного анализа сигналов",
        style='List Bullet'
    )
    doc.add_paragraph(
        "• Анализировать мощностные спектры для поиска доминирующих частот и периодичностей",
        style='List Bullet'
    )
    doc.add_paragraph(
        "• Построить диаграмму Ляпунова для визуализации зависимости динамики от параметров",
        style='List Bullet'
    )

    doc.add_heading('8. Примеры сгенерированных последовательностей', level=1)
    seq_str = ''.join(seq1[:300])
    p = doc.add_paragraph(f"Первые 300 символов из G1:\n{seq_str}")
    p.paragraph_format.left_indent = int(0.5 * 914400)
    doc.add_paragraph(f"Всего символов сгенерировано: {len(seq1):,}")

    filename = os.path.join(
        OUTPUT_DIR,
        f"{prefix}_{datetime.now().strftime('%d.%m.%Y_%H_%M_%S')}.docx"
    )
    doc.save(filename)
    return filename


def export_to_pdf(seq, fig_3d, fig_phase, fig_noise, fig_hist, fig_dyn,
                  fig_sync_point, fig_tm, fig_spec, fig_runs, prefix="sequence"):
    figures = []
    figure_names = []

    if fig_3d is not None:
        figures.append(fig_3d)
        figure_names.append("3D Attractors")
    if fig_phase is not None:
        figures.append(fig_phase)
        figure_names.append("Phase Trajectories")
    if fig_noise is not None:
        figures.append(fig_noise)
        figure_names.append("Deterministic Noise (X-component)")
    if fig_hist is not None:
        figures.append(fig_hist)
        figure_names.append("Symbol Histogram (G1)")
    if fig_dyn is not None:
        figures.append(fig_dyn)
        figure_names.append("Match Dynamics")
    if fig_sync_point is not None:
        figures.append(fig_sync_point)
        figure_names.append("Synchronization Point")
    if fig_tm is not None:
        figures.append(fig_tm)
        figure_names.append("Transition Matrix (G1)")
    if fig_spec is not None:
        figures.append(fig_spec)
        figure_names.append("Spectrum X-component (G1)")
    if fig_runs is not None:
        figures.append(fig_runs)
        figure_names.append("Symbol Run Lengths (G1)")

    pdf = FPDF(orientation='P', unit='mm', format='A4')

    pdf.add_page()
    pdf.set_font("Arial", size=16, style='B')
    pdf.cell(0, 20, "Lorenz Chaos Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Generated: {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}", ln=True, align='C')
    pdf.cell(0, 10, f"Total Symbols: {len(seq)}", ln=True, align='C')
    pdf.ln(10)
    pdf.cell(0, 10, "Analysis Graphs", ln=True, align='C')

    temp_dir = tempfile.mkdtemp()

    try:
        for i, (fig, name) in enumerate(zip(figures, figure_names)):
            try:
                temp_img = os.path.join(temp_dir, f"chart_{i}.png")
                fig.write_image(temp_img, width=800, height=600)

                pdf.add_page()
                pdf.set_font("Arial", size=12, style='B')
                pdf.cell(0, 10, name, ln=True)
                pdf.ln(5)
                pdf.image(temp_img, x=10, y=pdf.get_y(), w=190)
            except Exception as e:
                print(f"Warning: Could not add graph {name}: {e}")
                continue
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

    filename = os.path.join(
        OUTPUT_DIR,
        f"{prefix}_{datetime.now().strftime('%d.%m.%Y_%H_%M_%S')}.pdf"
    )
    pdf.output(filename)
    return filename


# -------- конец функций экспорта --------

def sliding_match_fraction(seqA, seqB, window=200, step=50, dt=0.05):
    minlen = min(len(seqA), len(seqB))
    times = []
    fracs = []
    if minlen < window:
        return np.array([]), np.array([])
    for start in range(0, minlen - window + 1, step):
        a = seqA[start:start + window]
        b = seqB[start:start + window]
        frac = sum(1 for i in range(window) if a[i] == b[i]) / window
        fracs.append(frac)
        times.append((start + window // 2) * dt)
    return np.array(times), np.array(fracs)


def symbol_repeat_stats(seq, symbol, dt=0.05):
    idxs = [i for i, c in enumerate(seq) if c == symbol]
    if len(idxs) < 2:
        return None
    intervals = np.diff(idxs) * dt
    return {
        'count': len(idxs),
        'mean_interval_s': float(np.mean(intervals)),
        'std_interval_s': float(np.std(intervals)),
        'intervals': intervals
    }


def analyze_sync_time(seq1, seq2, window=10):
    min_length = min(len(seq1), len(seq2))
    for i in range(min_length - window):
        if seq1[i:i + window] == seq2[i:i + window]:
            return i
    return None


def phase_difference(p1, p2):
    diff = np.abs(p1 - p2) % (2 * np.pi)
    return np.minimum(diff, 2 * np.pi - diff)


def detect_sync_stable_index(xyz1, xyz2):
    x1, y1, z1 = xyz1
    x2, y2, z2 = xyz2
    phase1 = np.arctan2(y1, z1)
    phase2 = np.arctan2(y2, z2)
    n = len(phase1)
    stable_points = []
    for i in range(0, n - WINDOW, SYNC_DETECT_STEP):
        dist = np.sqrt(
            (x1[i:i + WINDOW] - x2[i:i + WINDOW]) ** 2 +
            (y1[i:i + WINDOW] - y2[i:i + WINDOW]) ** 2 +
            (z1[i:i + WINDOW] - z2[i:i + WINDOW]) ** 2
        )
        phase_diff = phase_difference(phase1[i:i + WINDOW], phase2[i:i + WINDOW])
        if np.mean(dist) < 1.0 and np.mean(phase_diff) < PHASE_THRESH:
            stable_points.append(i)
    if not stable_points:
        return None
    return stable_points[-1] + WINDOW


def match_dynamics(seq1, seq2, window=20):
    length = min(len(seq1), len(seq2))
    matches = []
    for i in range(length - window + 1):
        window1 = seq1[i:i + window]
        window2 = seq2[i:i + window]
        match_count = sum(c1 == c2 for c1, c2 in zip(window1, window2))
        matches.append(match_count)
    return matches


def sequence_entropy(seq):
    vals, counts = np.unique(seq, return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def transition_matrix(seq, alphabet):
    idx_map = {ch: i for i, ch in enumerate(alphabet)}
    n = len(alphabet)
    M = np.zeros((n, n), dtype=float)
    for a, b in zip(seq[:-1], seq[1:]):
        if a in idx_map and b in idx_map:
            i, j = idx_map[a], idx_map[b]
            M[i, j] += 1
    row_sums = M.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    M /= row_sums
    return M


def run_lengths(seq, symbol):
    lengths = []
    cur = 0
    for c in seq:
        if c == symbol:
            cur += 1
        elif cur > 0:
            lengths.append(cur)
            cur = 0
    if cur > 0:
        lengths.append(cur)
    return lengths


def build_spectrum_figure(signal, dt, title="Амплитудный спектр"):
    sig = np.asarray(signal)
    n = len(sig)
    freqs = np.fft.rfftfreq(n, d=dt)
    fft_vals = np.fft.rfft(sig - sig.mean())
    amp = np.abs(fft_vals)
    fig = go.Figure(
        data=go.Scatter(x=freqs, y=amp, mode='lines')
    )
    fig.update_layout(
        title=title,
        xaxis_title="Частота, Гц",
        yaxis_title="Амплитуда",
        template="plotly",
        height=300
    )
    return fig


# -------------------------------------------------
# Фигуры Plotly
# -------------------------------------------------
def build_attractor_3d_figure(xyz1, xyz2, split_idx=None, title="Аттракторы"):
    x1, y1, z1 = xyz1
    x2, y2, z2 = xyz2
    fig = go.Figure()
    if split_idx is None:
        fig.add_trace(go.Scatter3d(x=x1, y=y1, z=z1, mode='lines', name='Генератор 1'))
        fig.add_trace(go.Scatter3d(x=x2, y=y2, z=z2, mode='lines', name='Генератор 2'))
    else:
        fig.add_trace(go.Scatter3d(
            x=x1[:split_idx], y=y1[:split_idx], z=z1[:split_idx],
            mode='lines', name='G1 до'
        ))
        fig.add_trace(go.Scatter3d(
            x=x1[split_idx:], y=y1[split_idx:], z=z1[split_idx:],
            mode='lines', name='G1 после'
        ))
        fig.add_trace(go.Scatter3d(
            x=x2[:split_idx], y=y2[:split_idx], z=z2[:split_idx],
            mode='lines', name='G2 до'
        ))
        fig.add_trace(go.Scatter3d(
            x=x2[split_idx:], y=y2[split_idx:], z=z2[split_idx:],
            mode='lines', name='G2 после'
        ))
    fig.update_layout(title=title, height=800, template="plotly")
    return fig


def build_phase_figure(xyz1, xyz2, dt, split_idx=None, title="Фазы"):
    x1, y1, z1 = xyz1
    x2, y2, z2 = xyz2
    phase1 = np.arctan2(y1, z1)
    phase2 = np.arctan2(y2, z2)
    t = np.arange(len(phase1)) * dt
    fig = go.Figure()
    if split_idx is None:
        fig.add_trace(go.Scatter(x=t, y=phase1, mode='lines', name='Фаза 1'))
        fig.add_trace(go.Scatter(x=t, y=phase2, mode='lines', name='Фаза 2'))
    else:
        fig.add_trace(go.Scatter(
            x=t[:split_idx], y=phase1[:split_idx],
            mode='lines', name='Фаза1 до'
        ))
        fig.add_trace(go.Scatter(
            x=t[split_idx:], y=phase1[split_idx:],
            mode='lines', name='Фаза1 после'
        ))
        fig.add_trace(go.Scatter(
            x=t[:split_idx], y=phase2[:split_idx],
            mode='lines', name='Фаза2 до'
        ))
        fig.add_trace(go.Scatter(
            x=t[split_idx:], y=phase2[split_idx:],
            mode='lines', name='Фаза2 после'
        ))
    fig.update_layout(title=title, height=350, xaxis_title='t (s)', template="plotly")
    return fig


def build_noise_figure(signal, title="Детерминированный шум (Lorenz)"):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=signal, mode='lines', name='noise'))
    fig.update_layout(
        title=title, height=300,
        xaxis_title='Итерация', yaxis_title='Амплитуда',
        template="plotly"
    )
    return fig


def build_histogram_figure(seq, title="Гистограмма символов"):
    df = pd.DataFrame(seq, columns=["symbol"])
    freq = df["symbol"].value_counts().sort_index()
    fig = go2.Figure([go2.Bar(x=freq.index, y=freq.values)])
    fig.update_layout(
        title=title,
        xaxis_title="Символ",
        yaxis_title="Частота",
        height=350,
        template="plotly"
    )
    return fig


def build_match_dynamics_figure(seq1, seq2, window=20, title="Динамика совпадений"):
    matches = match_dynamics(seq1, seq2, window=window)
    x_vals = list(range(len(matches)))
    fig = go2.Figure([go2.Scatter(x=x_vals, y=matches, mode='lines+markers')])
    fig.update_layout(
        title=f"{title} (окно={window})",
        xaxis_title="Позиция окна",
        yaxis_title="Совпадений",
        height=300,
        template="plotly"
    )
    return fig


def build_sync_point_figure(time_arr, frac_times, frac_vals, sync_time=None,
                            title="Точка синхронизации / совпадения символов"):
    fig = go.Figure()
    if len(frac_times):
        fig.add_trace(go.Scatter(
            x=frac_times, y=frac_vals * 100,
            mode='lines+markers', name='Совпадения (%)'
        ))
    if sync_time is not None:
        fig.add_trace(go.Scatter(
            x=[sync_time],
            y=[100],
            mode='markers',
            marker=dict(size=12, symbol='x', color='red'),
            name='Полная синхр.'
        ))
    fig.update_layout(
        title=title,
        xaxis_title='Время, с',
        yaxis_title='Совпадения в окне, %',
        height=350,
        template="plotly"
    )
    return fig

# -------------------------------------------------
# Dash приложение
# -------------------------------------------------
alphabet_options = [
    {'label': 'Только цифры', 'value': 'digits'},
    {'label': 'Только латиница', 'value': 'latin'},
    {'label': 'Только кириллица', 'value': 'cyrillic'},
    {'label': 'Таблица ASCII', 'value': 'ascii'},
    {'label': 'Unicode (печатаемые)', 'value': 'unicode'},
    {'label': 'EBCDIC', 'value': 'ebcdic'},
    {'label': 'KOI-8', 'value': 'koi8'},
    {'label': 'Только спецсимволы', 'value': 'special'},
]

app = dash.Dash(__name__, suppress_callback_exceptions=True)

# --- Шапка: параметры + вкладки + кнопки в одну строку ---
controls_bar = html.Div(
    [
        html.Div("Lorenz Chaos Lab",
                 style={"fontWeight": "bold", "marginRight": "20px"}),

        html.Label("Алфавит", style={"marginRight": "5px"}),
        dcc.Dropdown(
            options=alphabet_options,
            value='ascii',
            id='alphabet-dropdown',
            style={"width": "160px", "marginRight": "15px"},
        ),

        html.Label("T_MAX", style={"marginRight": "5px"}),
        dcc.Input(
            id='param-tmax',
            type='number',
            placeholder=str(T_MAX),
            value=None,
            style={"width": "90px", "marginRight": "15px"},
        ),

        html.Label("DT", style={"marginRight": "5px"}),
        dcc.Input(
            id='param-dt',
            type='number',
            placeholder=str(DT),
            value=None,
            style={"width": "80px", "marginRight": "15px"},
        ),

        html.Label("R_RATIO", style={"marginRight": "5px"}),
        dcc.Input(
            id='param-r',
            type='number',
            placeholder=str(R_RATIO),
            value=None,
            style={"width": "80px", "marginRight": "15px"},
        ),

        html.Label("Символ", style={"marginRight": "5px"}),
        dcc.Input(
            id='symbol-input',
            value='A',
            type='text',
            maxLength=1,
            style={"width": "60px", "marginRight": "20px"},
        ),

        dcc.Tabs(
            id="tabs",
            value='app3',
            children=[
                dcc.Tab(label='dual run', value='app3'),
                dcc.Tab(label='coupled', value='app4'),
            ],
            style={"marginRight": "20px"},
        ),

        html.Button("Пересчитать", id="analyze-btn",
                    style={"marginRight": "10px"}),
        html.Button("Excel", id="export-excel-btn",
                    style={"marginRight": "10px"}),
        html.Button("PDF", id="export-pdf-btn",
                    style={"marginRight": "10px"}),
        html.Button("Word", id="export-word-btn"),

        html.Div(
            id="save-msg",
            style={"marginLeft": "15px", "fontSize": "0.9rem"},
        ),
    ],
    style={
        "display": "flex",
        "flexWrap": "wrap",
        "alignItems": "center",
        "gap": "4px",
        "padding": "8px 12px",
        "borderBottom": "1px solid #ccc",
        "backgroundColor": "white",
    },
)

app.layout = html.Div(
    [
        controls_bar,
        html.Div(id="tab-content"),
    ],
    style={
        "backgroundColor": "white",
        "color": "black",
        "padding": "0",
        "margin": "0",
        "boxSizing": "border-box",
        "width": "100vw",
    },
)

# -------------------------------------------------
# Callback с корректной логикой экспорта
# -------------------------------------------------
@app.callback(
    Output('tab-content', 'children'),
    Output('save-msg', 'children'),
    Input('tabs', 'value'),
    Input('analyze-btn', 'n_clicks'),
    Input('export-excel-btn', 'n_clicks'),
    Input('export-pdf-btn', 'n_clicks'),
    Input('export-word-btn', 'n_clicks'),
    State('alphabet-dropdown', 'value'),
    State('symbol-input', 'value'),
    State('param-tmax', 'value'),
    State('param-dt', 'value'),
    State('param-r', 'value'),
    prevent_initial_call=False
)
def render_tab_content(tab_value, analyze_clicks,
                       export_excel_clicks, export_pdf_clicks, export_word_clicks,
                       selected_alphabet, symbol_of_interest,
                       tmax_user, dt_user, r_user):

    analyze_clicks = analyze_clicks or 0
    export_excel_clicks = export_excel_clicks or 0
    export_pdf_clicks = export_pdf_clicks or 0
    export_word_clicks = export_word_clicks or 0

    t_max = tmax_user if tmax_user not in (None, '') else T_MAX
    dt = dt_user if dt_user not in (None, '') else DT
    r_ratio = r_user if r_user not in (None, '') else R_RATIO

    alphabet = select_alphabet(selected_alphabet)

    if tab_value == 'app3':
        xyz1, xyz2 = dual_run(t_max, dt, r=None, phase_offset=0.4)
        split_idx = None
        title_prefix = "App3 — dual_run"
        prefix = "seq_app3"
        mode_desc = (
            "Dual Run (две независимые системы Лоренца "
            "с разными начальными условиями)"
        )
    else:
        a = (0.1, 0.0, 0.0)
        b = (0.15, 0.05, 0.05)
        xyz1, xyz2 = lorenz_coupled(t_max, dt, *a, *b, r_ratio)
        sync_idx_local = detect_sync_stable_index(xyz1, xyz2)
        split_idx = sync_idx_local if sync_idx_local is not None else None
        title_prefix = "App4 — coupled"
        prefix = "seq_app4"
        mode_desc = (
            f"Coupled Mode (две связанные системы Лоренца "
            f"с коэффициентом связи R={r_ratio})"
        )

    time_arr_local = np.arange(len(xyz1[0])) * dt

    fig_3d = build_attractor_3d_figure(
        xyz1, xyz2, split_idx=split_idx,
        title=f"{title_prefix}: Аттрактор 3D"
    )
    fig_phase = build_phase_figure(
        xyz1, xyz2, dt=dt, split_idx=split_idx,
        title=f"{title_prefix}: Фазы"
    )

    x_signal = xyz1[0]
    signal = (x_signal - np.mean(x_signal)) / np.std(x_signal)
    fig_noise = build_noise_figure(
        signal,
        title=f"{title_prefix}: Детерминированный шум (x-компонента)"
    )

    seq1 = chaos_signal_to_sequence(x_signal, alphabet)
    seq2 = chaos_signal_to_sequence(xyz2[0], alphabet)

    fig_hist = build_histogram_figure(
        seq1,
        title=f"{title_prefix}: Гистограмма символов (G1)"
    )
    dyn_window = min(50, max(5, WINDOW // 10))
    fig_dyn = build_match_dynamics_figure(
        seq1, seq2, window=dyn_window,
        title=f"{title_prefix}: Динамика совпадений"
    )

    times, fracs = sliding_match_fraction(seq1, seq2, window=WINDOW,
                                          step=STEP, dt=dt)
    mean_match = float(np.mean(fracs) * 100) if len(fracs) else 0.0

    stat1 = symbol_repeat_stats(seq1, symbol_of_interest, dt)
    stat2 = symbol_repeat_stats(seq2, symbol_of_interest, dt)
    idx_full_sync = analyze_sync_time(
        seq1, seq2,
        window=WINDOW if WINDOW < len(seq1) else 10
    )
    sync_time_s = (idx_full_sync * dt) if idx_full_sync is not None else None

    fig_sync_point = build_sync_point_figure(
        time_arr_local, times, fracs,
        sync_time=sync_time_s,
        title=f"{title_prefix}: Совпадения и точка полной синхронизации"
    )

    H1 = sequence_entropy(seq1)
    H2 = sequence_entropy(seq2)

    M = transition_matrix(seq1, alphabet)
    fig_tm = go.Figure(
        data=go.Heatmap(
            z=M,
            x=list(alphabet),
            y=list(alphabet),
            colorscale='Viridis',
            zmin=0,
            zmax=1,
            colorbar_title='P(j|i)'
        )
    )
    fig_tm.update_layout(
        title=f"{title_prefix}: Матрица переходов (G1)",
        xaxis_title="Следующий символ",
        yaxis_title="Текущий символ",
        template="plotly",
        height=400
    )

    runs1 = run_lengths(seq1, symbol_of_interest)
    fig_runs = None
    if runs1:
        fig_runs = go.Figure(
            data=go.Histogram(x=runs1, nbinsx=20)
        )
        fig_runs.update_layout(
            title=(
                f"{title_prefix}: Длины подряд идущего символа "
                f"'{symbol_of_interest}' (G1)"
            ),
            xaxis_title="Длина серии",
            yaxis_title="Частота",
            template="plotly",
            height=300
        )

    fig_spec = build_spectrum_figure(
        x_signal, dt,
        title=f"{title_prefix}: Спектр x-компоненты"
    )

    stat1_str = (
        f"count={stat1['count']}, "
        f"mean={stat1['mean_interval_s']:.4f} c, "
        f"std={stat1['std_interval_s']:.4f} c"
        if stat1 else "недостаточно повторов"
    )
    stat2_str = (
        f"count={stat2['count']}, "
        f"mean={stat2['mean_interval_s']:.4f} c, "
        f"std={stat2['std_interval_s']:.4f} c"
        if stat2 else "недостаточно повторов"
    )

    results_card = html.Div(
        [
            html.H3("Результаты анализа"),
            html.P(
                f"Средняя доля совпадений (скользящее окно): "
                f"{mean_match:.2f} %"
            ),
            html.P(
                f"Полная синхронизация последовательностей обнаружена "
                f"примерно в {sync_time_s:.2f} с"
                if sync_time_s is not None else
                "Полная синхронизация последовательностей не обнаружена"
            ),
            html.P(f"Энтропия G1: {H1:.3f} бит/символ"),
            html.P(f"Энтропия G2: {H2:.3f} бит/символ"),
            html.P(
                f"Статистика повтора символа '{symbol_of_interest}':"
            ),
            html.Ul(
                [
                    html.Li(f"G1: {stat1_str}"),
                    html.Li(f"G2: {stat2_str}"),
                ]
            ),
            html.P(
                f"Параметры моделирования: T_MAX={t_max}, "
                f"DT={dt}, R_RATIO={r_ratio}"
            ),
        ],
        style={"marginTop": "20px"},
    )

    # --- каждый график на отдельной строке ---
    graphs_layout_children = [
        html.Div(dcc.Graph(figure=fig_3d),         style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_phase),      style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_noise),      style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_hist),       style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_dyn),        style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_sync_point), style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_tm),         style={"marginBottom": "20px"}),
        html.Div(dcc.Graph(figure=fig_spec),       style={"marginBottom": "20px"}),
    ]

    if fig_runs is not None:
        graphs_layout_children.append(
            html.Div(dcc.Graph(figure=fig_runs),
                     style={"marginBottom": "20px"})
        )

    graphs_layout_children.append(results_card)
    graphs_layout = html.Div(graphs_layout_children)

    save_msg_parts = []

    ctx = callback_context
    triggered_id = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None

    if triggered_id == 'analyze-btn' and analyze_clicks:
        filename_csv, length = save_sequence_csv(seq1, prefix=prefix)
        save_msg_parts.append(f"✓ CSV: {length:,} символов → {filename_csv}")

    if triggered_id == 'export-excel-btn' and export_excel_clicks:
        fname_xlsx = export_to_excel(
            seq1, t_max, dt, r_ratio, H1, H2, mean_match, sync_time_s,
            prefix=prefix
        )
        save_msg_parts.append(f"✓ Excel: {fname_xlsx}")

    if triggered_id == 'export-pdf-btn' and export_pdf_clicks:
        try:
            fname_pdf = export_to_pdf(
                seq1, fig_3d, fig_phase, fig_noise, fig_hist, fig_dyn,
                fig_sync_point, fig_tm, fig_spec, fig_runs, prefix=prefix
            )
            save_msg_parts.append(f"✓ PDF графики: {fname_pdf}")
        except Exception as e:
            save_msg_parts.append(f"✗ PDF ошибка: {str(e)[:80]}")

    if triggered_id == 'export-word-btn' and export_word_clicks:
        try:
            fname_docx = export_to_word_detailed(
                seq1, seq2, stat1, stat2, H1, H2,
                t_max, dt, r_ratio, mean_match, sync_time_s,
                mode_desc, symbol_of_interest, len(alphabet),
                prefix=prefix
            )
            save_msg_parts.append(f"✓ Word отчёт: {fname_docx}")
        except Exception as e:
            save_msg_parts.append(f"✗ Word ошибка: {str(e)[:80]}")

    save_msg = " | ".join(save_msg_parts) if save_msg_parts else ""

    return graphs_layout, save_msg


# -------------------------------------------------
# Запуск приложения
# -------------------------------------------------
if __name__ == '__main__':
    # pip install dash dash-bootstrap-components plotly kaleido pandas numpy
    # python-docx fpdf openpyxl
    app.run(
    mode="inline",        # или "jupyterlab" / "external"
    width="100%",         # ширина iframe
    height=2100,           # высота iframe (px) – поставь, сколько нужно
)


In [31]:
# ЯЧЕЙКА ДЛЯ JUPYTER: визуализации Лоренца (все графики 2D, с защитой dt)

import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown, RadioButtons, FloatSlider, VBox, HBox, fixed

# -------------------------------------------------
# 1. Классическая система Лоренца (математическая модель)
# -------------------------------------------------

def lorenz_ode(state, t, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return dx, dy, dz

def integrate_lorenz(t_max=40.0, dt=0.01,
                     sigma=10.0, rho=28.0, beta=8.0/3.0,
                     x0=1.0, y0=1.0, z0=1.0):
    # защита от отрицательного шага
    dt = abs(dt)
    n = max(int(t_max / dt), 2)
    t = np.linspace(0.0, t_max, n)
    x = np.zeros(n)
    y = np.zeros(n)
    z = np.zeros(n)
    x[0], y[0], z[0] = x0, y0, z0
    for i in range(1, n):
        dx, dy, dz = lorenz_ode((x[i-1], y[i-1], z[i-1]), t[i-1],
                                sigma=sigma, rho=rho, beta=beta)
        x[i] = x[i-1] + dx*dt
        y[i] = y[i-1] + dy*dt
        z[i] = z[i-1] + dz*dt
    return t, x, y, z

def calculate_spectrum(signal, dt):
    sig = np.asarray(signal)
    n = len(sig)
    dt = abs(dt)
    freqs = np.fft.rfftfreq(n, d=dt)
    fft_vals = np.fft.rfft(sig - sig.mean())
    amp = np.abs(fft_vals)
    return freqs, amp

# -------------------------------------------------
# 2. Аналоговая система + стабилизированная интеграция
# -------------------------------------------------

R1 = 100000.0
R2 = 100000.0
R3 = 35700.0
R4 = 1000000.0
R8 = 250000.0
R9 = 250000.0
R10 = 133.0
C1 = 1e-9
C2 = 1e-9
C4 = 3e-9

def analog_lorenz_ode(state, t):
    x, y, z = state
    x_dt = (1 / (R1 * C1)) * (y - x)
    y_dt = (1 / (R3 * C2)) * ((R4 / R3) * x - y - z * x)
    z_dt = (1 / (R9 * C4)) * (-z + (x * y) / (R8 * R10))
    return x_dt, y_dt, z_dt

def integrate_analog_stable(t_max=0.02, dt=0.0001,
                             x0=0.1, y0=0.1, z0=0.1,
                             clip=100.0):
    dt = abs(dt)
    n = max(int(t_max / dt), 2)
    t = np.linspace(0.0, t_max, n)
    x = np.zeros(n)
    y = np.zeros(n)
    z = np.zeros(n)
    x[0], y[0], z[0] = x0, y0, z0
    for i in range(1, n):
        dx, dy, dz = analog_lorenz_ode((x[i-1], y[i-1], z[i-1]), t[i-1])
        x[i] = np.clip(x[i-1] + dx*dt, -clip, clip)
        y[i] = np.clip(y[i-1] + dy*dt, -clip, clip)
        z[i] = np.clip(z[i-1] + dz*dt, -clip, clip)
        if not np.all(np.isfinite([x[i], y[i], z[i]])):
            x = x[:i]
            y = y[:i]
            z = z[:i]
            t = t[:i]
            break
    return t, x, y, z

# -------------------------------------------------
# 3. Универсальная функция построения 2D-графика
# -------------------------------------------------

def make_figure(
    graph_type,
    sigma, rho, beta,
    x0, y0, z0,
    t_max_math, dt_math,
    t_max_analog, dt_analog,
    x_axis_choice, y_axis_choice,
    portrait_type
):
    # Математическая модель
    t_m, x_m, y_m, z_m = integrate_lorenz(
        t_max=t_max_math, dt=dt_math,
        sigma=sigma, rho=rho, beta=beta,
        x0=x0, y0=y0, z0=z0
    )

    # Аналоговая модель (стабильная интеграция)
    t_a, x_a, y_a, z_a = integrate_analog_stable(
        t_max=t_max_analog, dt=dt_analog,
        x0=min(x0, 0.2), y0=min(y0, 0.2), z0=min(z0, 0.2),
        clip=100.0
    )

    fig = go.Figure()

    # ----- 2D аттракторы / проекции -----
    if graph_type == 'attr_math_xy':
        fig.add_trace(go.Scatter(x=x_m, y=y_m, mode='lines',
                                 line=dict(color='blue'), name='Math x-y'))
        fig.update_layout(title='2D аттрактор Лоренца (Math, x-y)',
                          xaxis_title='x', yaxis_title='y', height=500)

    elif graph_type == 'attr_math_xz':
        fig.add_trace(go.Scatter(x=x_m, y=z_m, mode='lines',
                                 line=dict(color='blue'), name='Math x-z'))
        fig.update_layout(title='2D аттрактор Лоренца (Math, x-z)',
                          xaxis_title='x', yaxis_title='z', height=500)

    elif graph_type == 'attr_math_yz':
        fig.add_trace(go.Scatter(x=y_m, y=z_m, mode='lines',
                                 line=dict(color='blue'), name='Math y-z'))
        fig.update_layout(title='2D аттрактор Лоренца (Math, y-z)',
                          xaxis_title='y', yaxis_title='z', height=500)

    elif graph_type == 'attr_analog_xy':
        fig.add_trace(go.Scatter(x=x_a, y=y_a, mode='lines',
                                 line=dict(color='red'), name='Analog x-y'))
        fig.update_layout(title='2D аттрактор Лоренца (Analog, x-y)',
                          xaxis_title='x', yaxis_title='y', height=500)

    elif graph_type == 'attr_analog_xz':
        fig.add_trace(go.Scatter(x=x_a, y=z_a, mode='lines',
                                 line=dict(color='red'), name='Analog x-z'))
        fig.update_layout(title='2D аттрактор Лоренца (Analog, x-z)',
                          xaxis_title='x', yaxis_title='z', height=500)

    elif graph_type == 'attr_analog_yz':
        fig.add_trace(go.Scatter(x=y_a, y=z_a, mode='lines',
                                 line=dict(color='red'), name='Analog y-z'))
        fig.update_layout(title='2D аттрактор Лоренца (Analog, y-z)',
                          xaxis_title='y', yaxis_title='z', height=500)

    elif graph_type == '2d_projection_math':
        data_map = {'x': x_m, 'y': y_m, 'z': z_m}
        fig.add_trace(go.Scatter(
            x=data_map[x_axis_choice],
            y=data_map[y_axis_choice],
            mode='lines',
            line=dict(color='blue'),
            name=f'{x_axis_choice.upper()}-{y_axis_choice.upper()}'
        ))
        fig.update_layout(
            title=f'2D проекция (Math): {x_axis_choice.upper()} vs {y_axis_choice.upper()}',
            xaxis_title=x_axis_choice.upper(),
            yaxis_title=y_axis_choice.upper(),
            height=500
        )

    elif graph_type == '2d_projection_analog':
        data_map = {'x': x_a, 'y': y_a, 'z': z_a}
        fig.add_trace(go.Scatter(
            x=data_map[x_axis_choice],
            y=data_map[y_axis_choice],
            mode='lines',
            line=dict(color='red'),
            name=f'{x_axis_choice.upper()}-{y_axis_choice.upper()}'
        ))
        fig.update_layout(
            title=f'2D проекция (Analog): {x_axis_choice.upper()} vs {y_axis_choice.upper()}',
            xaxis_title=x_axis_choice.upper(),
            yaxis_title=y_axis_choice.upper(),
            height=500
        )

    # ----- Осциллограммы -----
    elif graph_type == 'uy_osc':
        fig.add_trace(go.Scatter(x=t_m, y=y_m, mode='lines',
                                 line=dict(color='blue'), name='Uy(t)'))
        fig.update_layout(title='Осциллограмма Uy(t) (Math)',
                          xaxis_title='Время', yaxis_title='Uy', height=500)

    elif graph_type == 'uz_osc':
        fig.add_trace(go.Scatter(x=t_m, y=z_m, mode='lines',
                                 line=dict(color='green'), name='Uz(t)'))
        fig.update_layout(title='Осциллограмма Uz(t) (Math)',
                          xaxis_title='Время', yaxis_title='Uz', height=500)

    # ----- Фазовые портреты (Math) -----
    elif graph_type == 'phase_portrait':
        if portrait_type == 'ux':
            fig.add_trace(go.Scatter(x=x_m, y=y_m, mode='lines',
                                     line=dict(color='blue', width=1), name='Ux (x-y)'))
            fig.add_trace(go.Scatter(x=[x_m[0]], y=[y_m[0]], mode='markers',
                                     marker=dict(color='red', size=8), name='Начальная точка'))
            fig.update_layout(title='Фазовый портрет Ux (x-y)',
                              xaxis_title='x', yaxis_title='y', height=500)
        elif portrait_type == 'uy':
            fig.add_trace(go.Scatter(x=y_m, y=z_m, mode='lines',
                                     line=dict(color='green', width=1), name='Uy (y-z)'))
            fig.add_trace(go.Scatter(x=[y_m[0]], y=[z_m[0]], mode='markers',
                                     marker=dict(color='red', size=8), name='Начальная точка'))
            fig.update_layout(title='Фазовый портрет Uy (y-z)',
                              xaxis_title='y', yaxis_title='z', height=500)
        else:  # uz
            fig.add_trace(go.Scatter(x=z_m, y=x_m, mode='lines',
                                     line=dict(color='purple', width=1), name='Uz (z-x)'))
            fig.add_trace(go.Scatter(x=[z_m[0]], y=[x_m[0]], mode='markers',
                                     marker=dict(color='red', size=8), name='Начальная точка'))
            fig.update_layout(title='Фазовый портрет Uz (z-x)',
                              xaxis_title='z', yaxis_title='x', height=500)

    # ----- Спектры -----
    elif graph_type == 'spectrum_math_x':
        freqs, amp = calculate_spectrum(x_m, dt_math)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='blue'), name='Ux (Math)'))
        fig.update_layout(title='Амплитудный спектр Ux (Math)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    elif graph_type == 'spectrum_math_y':
        freqs, amp = calculate_spectrum(y_m, dt_math)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='blue'), name='Uy (Math)'))
        fig.update_layout(title='Амплитудный спектр Uy (Math)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    elif graph_type == 'spectrum_math_z':
        freqs, amp = calculate_spectrum(z_m, dt_math)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='blue'), name='Uz (Math)'))
        fig.update_layout(title='Амплитудный спектр Uz (Math)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    elif graph_type == 'spectrum_analog_x':
        freqs, amp = calculate_spectrum(x_a, dt_analog)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='red'), name='Ux (Analog)'))
        fig.update_layout(title='Амплитудный спектр Ux (Analog)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    elif graph_type == 'spectrum_analog_y':
        freqs, amp = calculate_spectrum(y_a, dt_analog)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='red'), name='Uy (Analog)'))
        fig.update_layout(title='Амплитудный спектр Uy (Analog)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    elif graph_type == 'spectrum_analog_z':
        freqs, amp = calculate_spectrum(z_a, dt_analog)
        fig.add_trace(go.Scatter(x=freqs, y=amp, mode='lines',
                                 line=dict(color='red'), name='Uz (Analog)'))
        fig.update_layout(title='Амплитудный спектр Uz (Analog)',
                          xaxis_title='Частота, Гц', yaxis_title='Амплитуда', height=500)

    fig.show()

# -------------------------------------------------
# 4. Виджеты управления (только 2D)
# -------------------------------------------------

graph_type_widget = Dropdown(
    options=[
        ('2D аттрактор Math x-y', 'attr_math_xy'),
        ('2D аттрактор Math x-z', 'attr_math_xz'),
        ('2D аттрактор Math y-z', 'attr_math_yz'),
        ('2D аттрактор Analog x-y', 'attr_analog_xy'),
        ('2D аттрактор Analog x-z', 'attr_analog_xz'),
        ('2D аттрактор Analog y-z', 'attr_analog_yz'),
        ('2D проекция (Math)', '2d_projection_math'),
        ('2D проекция (Analog)', '2d_projection_analog'),
        ('Осциллограмма Uy(t)', 'uy_osc'),
        ('Осциллограмма Uz(t)', 'uz_osc'),
        ('Фазовый портрет Ux/Uy/Uz', 'phase_portrait'),
        ('Спектр Ux (Math)', 'spectrum_math_x'),
        ('Спектр Uy (Math)', 'spectrum_math_y'),
        ('Спектр Uz (Math)', 'spectrum_math_z'),
        ('Спектр Ux (Analog)', 'spectrum_analog_x'),
        ('Спектр Uy (Analog)', 'spectrum_analog_y'),
        ('Спектр Uz (Analog)', 'spectrum_analog_z'),
    ],
    value='attr_math_xy',
    description='График:'
)

x_axis_widget = Dropdown(
    options=[('X', 'x'), ('Y', 'y'), ('Z', 'z')],
    value='x',
    description='X-ось:'
)

y_axis_widget = Dropdown(
    options=[('X', 'x'), ('Y', 'y'), ('Z', 'z')],
    value='y',
    description='Y-ось:'
)

portrait_widget = RadioButtons(
    options=[
        ('Ux (x-y)', 'ux'),
        ('Uy (y-z)', 'uy'),
        ('Uz (z-x)', 'uz')
    ],
    value='ux',
    description='Портрет:'
)

sigma_widget = FloatSlider(description='σ', value=10.0, min=5.0, max=20.0, step=0.5)
rho_widget   = FloatSlider(description='ρ', value=28.0, min=10.0, max=40.0, step=1.0)
beta_widget  = FloatSlider(description='β', value=8.0/3.0, min=1.0, max=4.0, step=0.1)

tmax_math_widget   = FloatSlider(description='T math', value=40.0, min=5.0, max=60.0, step=1.0)
tmax_analog_widget = FloatSlider(description='T analog', value=0.02, min=0.005, max=0.05, step=0.005)

dt_math   = 0.01
dt_analog = 0.0001

ui_row1 = HBox([graph_type_widget])
ui_row2 = HBox([x_axis_widget, y_axis_widget, portrait_widget])
ui_row3 = HBox([sigma_widget, rho_widget, beta_widget])
ui_row4 = HBox([tmax_math_widget, tmax_analog_widget])

ui = VBox([ui_row1, ui_row2, ui_row3, ui_row4])

out = interact(
    make_figure,
    graph_type=graph_type_widget,
    sigma=sigma_widget,
    rho=rho_widget,
    beta=beta_widget,
    x0=fixed(0.1),
    y0=fixed(0.1),
    z0=fixed(0.1),
    t_max_math=tmax_math_widget,
    dt_math=fixed(dt_math),
    t_max_analog=tmax_analog_widget,
    dt_analog=fixed(dt_analog),
    x_axis_choice=x_axis_widget,
    y_axis_choice=y_axis_widget,
    portrait_type=portrait_widget
)

display(ui)

interactive(children=(Dropdown(description='График:', options=(('2D аттрактор Math x-y', 'attr_math_xy'), ('2D…